# Table 2 — lapse-rate soil simulations, +1.2°C

Uses the PPSA `hrrrak_lapserate` and `met_hrrrak_lapserate` **soil-enabled** profiles at the original **+1.2°C precipitation threshold**. The original cold/warm pit observations, target times, nearest-profile tolerance, thickness weighting, error calculations, rounding, and final two-model table layout are retained from `4_table2_lapserate.ipynb`.

The soil PRO files contain both soil and snow: 0501 gives N+1 boundary heights while density/temperature give N element values. The parser aligns those elements and includes only snow above the ground interface (0 cm); soil does not enter snow depth, SWE, bulk snow density, or mean snow temperature. Large profiles are streamed, retaining only the nearest pit profiles. The original notebook is unchanged. Results display inline without exports.


In [1]:
from pathlib import Path
import sys
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents]
    if (p/'sites/all_sites/_soil_profile_metrics.py').is_file())
sys.path.insert(0,str(ROOT/'sites/all_sites'))


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# =============================================================================
# USER SETTINGS
# =============================================================================

pro_paths = {
    "HRRR-AK Model": Path("/home/cassie/python/models/run_snowpack/sites/ppsa/output/hrrrak_lapserate_ppsa_WY2020-WY2025_soil.pro"),
    "Met Mixed Model": Path("/home/cassie/python/models/run_snowpack/sites/ppsa/output/met_hrrrak_lapserate_ppsa_WY2020-WY2025_soil.pro"),
}

pit_table = pd.DataFrame({
    "pit_name": ["“Cold” Pit", "“Warm” Pit"],
    "pit_time": ["2025-02-12 12:00", "2025-04-09 12:00"],
    "obs_HS_cm": [133, 170],
    "obs_bulk_density_kg_m3": [334, 401],
    "obs_mean_temperature_C": [-9.2, 0.0],
    "obs_SWE_mm": [444, 682],
})

# =============================================================================
# PARSE SNOWPACK .PRO FILE
# =============================================================================

from functools import lru_cache
from _soil_profile_metrics import nearest_profiles, snow_bulk_metrics

@lru_cache(maxsize=2)
def parse_snowpack_pro_profiles(path, codes=("0501", "0502", "0503")):
    return nearest_profiles(path, pit_table["pit_time"], tolerance="24h")

def profile_bulk_metrics(profile):
    return snow_bulk_metrics(profile)


def get_nearest_profile_metrics(profiles, target_time, tolerance="24h"):
    target_time = pd.to_datetime(target_time)
    times = pd.DatetimeIndex(profiles.keys())

    idx = times.get_indexer([target_time], method="nearest")[0]
    profile_time = times[idx]
    offset = abs(profile_time - target_time)

    if offset > pd.Timedelta(tolerance):
        raise ValueError(
            f"Nearest profile to {target_time} is {profile_time}, "
            f"outside tolerance {tolerance}"
        )

    metrics = profile_bulk_metrics(profiles[profile_time])
    metrics["profile_time"] = profile_time
    return metrics


# =============================================================================
# EXTRACT METRICS FROM BOTH MODEL FILES
# =============================================================================

rows = []

for _, pit in pit_table.iterrows():
    row = {
        "pit_name": pit["pit_name"],
        "pit_time": pd.to_datetime(pit["pit_time"]),
        "obs_HS_cm": pit["obs_HS_cm"],
        "obs_bulk_density_kg_m3": pit["obs_bulk_density_kg_m3"],
        "obs_mean_temperature_C": pit["obs_mean_temperature_C"],
        "obs_SWE_mm": pit["obs_SWE_mm"],
    }

    for model_name, pro_path in pro_paths.items():
        profiles = parse_snowpack_pro_profiles(pro_path)
        metrics = get_nearest_profile_metrics(
            profiles,
            pit["pit_time"],
            tolerance="24h"
        )

        prefix = (
            model_name
            .replace(" ", "_")
            .replace("-", "")
            .replace("/", "_")
        )

        row[f"{prefix}_HS_cm"] = metrics["HS_cm"]
        row[f"{prefix}_bulk_density_kg_m3"] = metrics["bulk_density_kg_m3"]
        row[f"{prefix}_mean_temperature_C"] = metrics["mean_temperature_C"]
        row[f"{prefix}_SWE_mm"] = metrics["SWE_mm"]
        row[f"{prefix}_profile_time"] = metrics["profile_time"]

    rows.append(row)

comparison = pd.DataFrame(rows)

# =============================================================================
# BUILD TABLE LIKE SCREENSHOT
# =============================================================================

model_cols = {
    "HRRR-AK Model": "HRRRAK_Model",
    "Met Mixed Model": "Met_Mixed_Model",
}

table_rows = []

for _, row in comparison.iterrows():

    # Main data row
    table_rows.append({
        ("Date", ""): row["pit_name"] + "\n" + row["pit_time"].strftime("%-d %b\n%Y"),

        ("Snow Depth\n(cm)", "Obs"): row["obs_HS_cm"],
        ("Snow Depth\n(cm)", "HRRR-AK\nModel"): row["HRRRAK_Model_HS_cm"],
        ("Snow Depth\n(cm)", "Met\nMixed\nModel"): row["Met_Mixed_Model_HS_cm"],

        ("Bulk Snow Density\n(kg m$^{-3}$)", "Obs"): row["obs_bulk_density_kg_m3"],
        ("Bulk Snow Density\n(kg m$^{-3}$)", "HRRR-AK\nModel"): row["HRRRAK_Model_bulk_density_kg_m3"],
        ("Bulk Snow Density\n(kg m$^{-3}$)", "Met\nMixed\nModel"): row["Met_Mixed_Model_bulk_density_kg_m3"],

        ("Mean Temperature\n(°C)", "Obs"): row["obs_mean_temperature_C"],
        ("Mean Temperature\n(°C)", "HRRR-AK\nModel"): row["HRRRAK_Model_mean_temperature_C"],
        ("Mean Temperature\n(°C)", "Met\nMixed\nModel"): row["Met_Mixed_Model_mean_temperature_C"],

        ("SWE\n(mm)(kg m$^{-2}$)", "Obs"): row["obs_SWE_mm"],
        ("SWE\n(mm)(kg m$^{-2}$)", "HRRR-AK\nModel"): row["HRRRAK_Model_SWE_mm"],
        ("SWE\n(mm)(kg m$^{-2}$)", "Met\nMixed\nModel"): row["Met_Mixed_Model_SWE_mm"],
    })

    # Error row
    table_rows.append({
        ("Date", ""): "Model -\nObs\nError (%)",

        ("Snow Depth\n(cm)", "Obs"): "",
        ("Snow Depth\n(cm)", "HRRR-AK\nModel"): 100 * (row["HRRRAK_Model_HS_cm"] - row["obs_HS_cm"]) / row["obs_HS_cm"],
        ("Snow Depth\n(cm)", "Met\nMixed\nModel"): 100 * (row["Met_Mixed_Model_HS_cm"] - row["obs_HS_cm"]) / row["obs_HS_cm"],

        ("Bulk Snow Density\n(kg m$^{-3}$)", "Obs"): "",
        ("Bulk Snow Density\n(kg m$^{-3}$)", "HRRR-AK\nModel"): 100 * (row["HRRRAK_Model_bulk_density_kg_m3"] - row["obs_bulk_density_kg_m3"]) / row["obs_bulk_density_kg_m3"],
        ("Bulk Snow Density\n(kg m$^{-3}$)", "Met\nMixed\nModel"): 100 * (row["Met_Mixed_Model_bulk_density_kg_m3"] - row["obs_bulk_density_kg_m3"]) / row["obs_bulk_density_kg_m3"],

        ("Mean Temperature\n(°C)", "Obs"): "",
        ("Mean Temperature\n(°C)", "HRRR-AK\nModel"): "",
        ("Mean Temperature\n(°C)", "Met\nMixed\nModel"): "",

        ("SWE\n(mm)(kg m$^{-2}$)", "Obs"): "",
        ("SWE\n(mm)(kg m$^{-2}$)", "HRRR-AK\nModel"): 100 * (row["HRRRAK_Model_SWE_mm"] - row["obs_SWE_mm"]) / row["obs_SWE_mm"],
        ("SWE\n(mm)(kg m$^{-2}$)", "Met\nMixed\nModel"): 100 * (row["Met_Mixed_Model_SWE_mm"] - row["obs_SWE_mm"]) / row["obs_SWE_mm"],
    })

table = pd.DataFrame(table_rows)
table.columns = pd.MultiIndex.from_tuples(table.columns)

# =============================================================================
# ROUNDING
# =============================================================================

table_display = table.copy()

snow_cols = [col for col in table_display.columns if "Snow Depth" in col[0]]
density_cols = [col for col in table_display.columns if "Density" in col[0]]
swe_cols = [col for col in table_display.columns if "SWE" in col[0]]
temp_cols = [col for col in table_display.columns if "Temperature" in col[0]]

error_rows = table_display[("Date", "")].astype(str).str.contains("Error", na=False)

def format_value(x, decimals=0):
    if pd.isna(x) or x == "":
        return ""
    return f"{float(x):.{decimals}f}"

# Whole numbers for snow depth, density, and SWE data rows
for col in snow_cols + density_cols + swe_cols:
    table_display[col] = table_display[col].apply(lambda x: format_value(x, decimals=0))

# One decimal for temperature data rows
for col in temp_cols:
    table_display[col] = table_display[col].apply(lambda x: format_value(x, decimals=1))

# Percent error rows: one decimal, but leave Obs columns blank
for col in snow_cols + density_cols + swe_cols:
    if col[1] == "Obs":
        table_display.loc[error_rows, col] = ""
    else:
        table_display.loc[error_rows, col] = table.loc[error_rows, col].apply(
            lambda x: format_value(x, decimals=1)
        )

# Leave temperature error row blank
for col in temp_cols:
    table_display.loc[error_rows, col] = ""

print(table_display.to_string(index=False))
from IPython.display import display
display(table_display)


                    Date Snow Depth\n(cm)                                  Bulk Snow Density\n(kg m$^{-3}$)                                  Mean Temperature\n(°C)                                  SWE\n(mm)(kg m$^{-2}$)                                 
                                      Obs HRRR-AK\nModel Met\nMixed\nModel                              Obs HRRR-AK\nModel Met\nMixed\nModel                    Obs HRRR-AK\nModel Met\nMixed\nModel                    Obs HRRR-AK\nModel Met\nMixed\nModel
“Cold” Pit\n12 Feb\n2025              133            139               132                              334            367               351                   -9.2           -5.2              -5.7                    444            509               463
 Model -\nObs\nError (%)                             4.4              -0.8                                             9.8               5.1                                                                                          14.7       

Date Snow Depth\n(cm)                                   \
                                         Obs HRRR-AK\nModel Met\nMixed\nModel   
0  “Cold” Pit\n12 Feb\n2025              133            139               132   
1   Model -\nObs\nError (%)                             4.4              -0.8   
2   “Warm” Pit\n9 Apr\n2025              170            192               184   
3   Model -\nObs\nError (%)                            13.0               8.3   

  Bulk Snow Density\n(kg m$^{-3}$)                                   \
                               Obs HRRR-AK\nModel Met\nMixed\nModel   
0                              334            367               351   
1                                             9.8               5.1   
2                              401            441               430   
3                                            10.0               7.3   

  Mean Temperature\n(°C)                                   \
                     Obs HRRR-AK\nModel Met\nMixed\nModel   
0                   -9.2           -5.2              -5.7   
1                                                           
2                    0.0           -0.0              -0.1   
3                                                           

  SWE\n(mm)(kg m$^{-2}$)                                   
                     Obs HRRR-AK\nModel Met\nMixed\nModel  
0                    444            509               463  
1                                  14.7               4.3  
2                    682            847               792  
3                                  24.2              16.1